In [261]:
import requests
from langchain_core.prompts import PromptTemplate 
from langchain_core.tools import tool, StructuredTool, InjectedToolArg
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from langchain.agents import create_agent
from pydantic import BaseModel, Field
from typing import Annotated
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

In [262]:
NEWS_DATA_API='pub_735de635d9114151a03f673ff50373bd'

In [263]:
#template to fetch relevant news articles by passing relevant query parameters to the API
prompt = PromptTemplate(
    template = """ 
        qInTitle:Search news articles for specific keywords or phrases present in the news titles only. The value must be URL-encoded and the maximum character limit permitted is 512 characters.\n
        qInMeta:Search news articles for specific keywords or phrases present in the news titles, URL, meta keywords and meta description only. The value must be URL-encoded and the maximum character limit permitted is 512 characters.\n
        q:Search news articles for specific keywords or phrases present in the news title, content, URL, meta keywords and meta description. The value must be URL-encoded and the maximum character limit permitted is 512 characters.\n
        timeframe:Search the news articles for a specific timeframe (Minutes and Hours). For hours, you can set a timeframe of 1 to 48, and for minutes, you can define a timeframe of 1m to 2880m. For example, if you want to get the news for the past 6 hours then use timeframe=6 and if you want to get news for the last 15 min then use timeframe=15m.\n
        category:Search news articles for a specific category. The available categories are business, entertainment, environment, food, health, politics, science, sports, technology and top.\n

        Note - You can only use timeframe either in hours or minutes.
        Note: qInTitle can't be used with q or qInMeta parameter in the same query.\n
        Note: qInMeta can't be used with q or qInTitle parameter in the same query.\n

        I need you to return the query parameters string to append to the API URL to fetch relevant news articles. 
        The query parameters should be in the format of key=value and separated by &. 
        Do not include the base URL or any other text. Only return the query parameters string.

        I need you to fetch news articles related to the {topic}.
""",

    input_variables = ["topic"]
)

In [264]:
model = ChatOllama(model="gemma4:e4b", temperature=0.7, num_ctx=8192,
                   system_message=SystemMessage(content="""You are a news assistant. 
CRITICAL RULE: You MUST execute tools sequentially. Never call multiple tools at the same time. 
You must receive the final output from `generate_query_params` before you are allowed to invoke `fetch_news_articles`."""))


In [265]:
parser = StrOutputParser()

In [266]:
class Article(BaseModel):
    title: str = Field(..., description="The title of the news article"),
    description: str = Field(..., description="A brief description of the news article"),
    source: str = Field(..., description="The source of the news article")

In [267]:
class ArticleList(BaseModel):
    articles: list[Article] = Field(description="List of 5 news articles")

In [268]:
articles_parser = PydanticOutputParser(pydantic_object=ArticleList)


In [269]:
@tool
def generate_query_params(topic: str):
    """Generate a formatted URL query parameter string for fetching news articles."""
    # print("generate_query_params called with topic:", topic)
    model = ChatOllama(model="gemma4:e4b", temperature=0.7, max_tokens=4000)
    chain = prompt | model | parser

    prompt_text = prompt.invoke({'topic': topic})
    print(prompt_text)

    query_params = chain.invoke({'topic' : topic})
    # print(query_params)

    return query_params

In [270]:
class NewsQueryParams(BaseModel):
    topic : str = Field(..., description="The topic to search news articles for")

# @tool
def fetch_news_articles(topic: str):
    """ Tool to fetch news articles. MUST only be called after generating query parameters. """

    query_chain = prompt | model | parser
    query_params = query_chain.invoke({"topic": topic})

    print("query_params_before_fetching: ",query_params)
    
    url = f"https://newsdata.io/api/1/latest?apikey={NEWS_DATA_API}&{query_params}&size=5"
    response = requests.get(url)
    data = response.json()
    
    # 1. Check for API success
    if data.get("status") == "success":
        clean_articles = []
        
        # 2. Extract only the essential text needed by the financial analyst prompt
        for article in data.get("results", []):
            clean_articles.append({
                "title": article.get("title"),
                "description": article.get("description"),
                "source": article.get("source_name")
            })
            
        # 3. Return the minimized list of dictionaries, NOT the raw JSON
        return clean_articles

    return data

fetch_news_articles_tool = StructuredTool.from_function(
    func=fetch_news_articles,
    name="fetch_news_articles",
    description="Fetch relevant news articles by passing relevant query parameters to the API",
    args_schema=NewsQueryParams
)

In [ ]:
agent = create_agent(
    model=model,
    tools=[fetch_news_articles_tool],
)

format_instructions = articles_parser.get_format_instructions()
prompt_text = f"""You are a financial analyst specializing in the Indian Stock Market (NSE/BSE).
Use the fetch_news_articles tool to get the latest market news, then identify 5 news events that might affect stocks listed on NSE or BSE.

Return the final answer strictly according to these formatting instructions:
{format_instructions}"""

response = agent.invoke({"messages": [HumanMessage(content=prompt_text)]})
raw_content = response["messages"][-1].content
# print("Raw content from agent:\n", raw_content)
structured_result = articles_parser.parse(raw_content)

for i, item in enumerate(structured_result.articles):
    print(f"Article {i+1}:\nTitle: {item.title}\nDescription: {item.description}\nSource: {item.source}\n")

c:\YASHIT\AI\market-announcements\.venv\Lib\site-packages\pydantic\json_schema.py:2463: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=True, description='The title of the news article'),) is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)
c:\YASHIT\AI\market-announcements\.venv\Lib\site-packages\pydantic\json_schema.py:2463: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=True, description='A brief description of the news article'),) is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)


query_params_before_fetching:  q=Indian%20Stock%20Market&category=business
Raw content from agent:
 ```json
{
  "articles": [
    {
      "title": "Indian Markets End 8th Straight Week Lower, Longest Losing Streak Since 2001; Nifty Falls 3.1%",
      "description": "Indian markets ended their eighth straight week lower, surpassing the seven-week Covid-era losing streak. Nifty fell 3.1% this week as FPI selling, oil and rupee pressure weighed.",
      "source": "News 18"
    },
    {
      "title": "D-Street logs longest spell of week slump in 25 years",
      "description": "Indian equity markets faced significant pressure as US bond yields surged, leading to a record selloff. The Sensex and Nifty experienced prolonged declines, surpassing previous downturns witnessed during the Covid pandemic. Foreign investors continued to withdraw their investments amidst rising crude oil prices, negatively impacting the rupee's value. Analysts suggest selective stock opportunities may arise despite